# 🤖 Aşama 2: Baseline Model (SVM & Naive Bayes)

**Eren Can'ın görevi — Klasik makine öğrenmesi modelleri**

Bu notebook'ta:
1. TF-IDF + Linear SVM eğitilecek
2. TF-IDF + Naive Bayes eğitilecek
3. Modeller karşılaştırılacak (accuracy, F1, confusion matrix)
4. En iyi model `models/` klasörüne kaydedilecek

> ⚠️ Önce **01_veri_kesfi_ve_onisleme.ipynb** çalıştırılmış olmalı (data/hepsiburada_clean.csv gerekli).

## 0) Colab Kurulumu

In [ ]:
import sys, os
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB and not os.path.exists('src'):
    !git clone https://github.com/erncanyildirim/metin-ozetleme-ve-duygu-analizi.git
    %cd metin-ozetleme-ve-duygu-analizi
    !pip install -q -r requirements.txt
    
    # Temizlenmiş veriyi 1. notebook'ta üretip Drive'a yedeklediyseniz buraya alın:
    # from google.colab import drive
    # drive.mount('/content/drive')
    # !cp /content/drive/MyDrive/hepsiburada_clean.csv data/
    
print('✅ Kurulum hazır')

## 1) Veriyi Yükle

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from src.baseline_model import load_and_prepare_data

# Veriyi yükle, ön işle, eğitim/test böl
X_train, X_test, y_train, y_test, label_names = load_and_prepare_data()

print(f'\n📊 Etiketler: {label_names}')
print(f'📊 Train boyutu: {len(X_train):,}')
print(f'📊 Test boyutu : {len(X_test):,}')

## 2) Modelleri Eğit ve Değerlendir

In [ ]:
from src.baseline_model import (
    create_svm_pipeline, create_nb_pipeline,
    train_model, evaluate_model, save_model,
)

# --- 2.1 Naive Bayes ---
nb_pipeline = create_nb_pipeline()
t0 = time.time()
nb_trained = train_model(nb_pipeline, X_train, y_train, 'Naive Bayes')
nb_time = time.time() - t0
nb_results = evaluate_model(nb_trained, X_test, y_test, label_names, 'Naive Bayes')
nb_results['train_time'] = nb_time
save_model(nb_trained, 'Naive Bayes')
print(f'\n⏱️  Naive Bayes eğitim süresi: {nb_time:.1f} sn')

In [ ]:
# --- 2.2 Linear SVM ---
svm_pipeline = create_svm_pipeline()
t0 = time.time()
svm_trained = train_model(svm_pipeline, X_train, y_train, 'SVM (LinearSVC)')
svm_time = time.time() - t0
svm_results = evaluate_model(svm_trained, X_test, y_test, label_names, 'SVM (LinearSVC)')
svm_results['train_time'] = svm_time
save_model(svm_trained, 'SVM (LinearSVC)')
print(f'\n⏱️  SVM eğitim süresi: {svm_time:.1f} sn')

## 3) Karşılaştırma Tablosu (Rapor Tablo 4.4.1)

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Naive Bayes', 'SVM (LinearSVC)'],
    'Accuracy': [nb_results['accuracy'], svm_results['accuracy']],
    'F1 (Macro)': [nb_results['f1_macro'], svm_results['f1_macro']],
    'F1 (Weighted)': [nb_results['f1_weighted'], svm_results['f1_weighted']],
    'Eğitim Süresi (sn)': [nb_time, svm_time],
})
comparison.set_index('Model')

In [ ]:
# Bar grafiği — F1 karşılaştırması
fig, ax = plt.subplots(figsize=(10, 6))
metrics_df = comparison.melt(
    id_vars='Model',
    value_vars=['Accuracy', 'F1 (Macro)', 'F1 (Weighted)'],
    var_name='Metrik',
    value_name='Skor',
)
sns.barplot(data=metrics_df, x='Metrik', y='Skor', hue='Model', palette=['#4a90e2', '#28a745'], ax=ax)
ax.set_title('Baseline Modeller — Performans Karşılaştırması', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=3)
plt.tight_layout()
plt.savefig('rapor/sekil_4_4_baseline_karsilastirma.png', dpi=150, bbox_inches='tight')
plt.show()

## 4) Confusion Matrix'ler (Şekil 4.4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, results) in zip(axes, [('Naive Bayes', nb_results), ('SVM', svm_results)]):
    cm = results['confusion_matrix']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=label_names, yticklabels=label_names, ax=ax, cbar=True)
    ax.set_title(f'{name} — Confusion Matrix', fontsize=12, fontweight='bold')
    ax.set_xlabel('Tahmin')
    ax.set_ylabel('Gerçek')

plt.tight_layout()
plt.savefig('rapor/sekil_4_4b_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 5) Hata Analizi — Yanlış Sınıflandırılan Örnekler

In [ ]:
# En iyi modelin (SVM) yanlış tahminlerine göz at
y_pred_svm = svm_results['predictions']
errors_df = pd.DataFrame({
    'metin': X_test.values,
    'gercek': y_test.values,
    'tahmin': y_pred_svm,
})
errors_df = errors_df[errors_df['gercek'] != errors_df['tahmin']]

print(f'❌ Toplam yanlış tahmin: {len(errors_df):,}')
print('\n📋 İlk 10 hata örneği:')
errors_df.head(10)

## 6) Tahmin Hızı Karşılaştırması

In [ ]:
# 1000 örnekte tahmin süresi
import time

sample = X_test.iloc[:1000]

t0 = time.time(); _ = nb_trained.predict(sample); nb_pred_time = time.time() - t0
t0 = time.time(); _ = svm_trained.predict(sample); svm_pred_time = time.time() - t0

print(f'⏱️  Tahmin Süresi (1000 yorum):')
print(f'   Naive Bayes : {nb_pred_time*1000:.1f} ms')
print(f'   SVM         : {svm_pred_time*1000:.1f} ms')

## 7) Demo Tahminler

In [ ]:
from src.baseline_model import predict_sentiment

demo_yorumlar = [
    'Bu ürünü kesinlikle tavsiye ederim, çok kaliteli',
    'Kargo geç geldi, ürün hasarlıydı, kötü deneyim',
    'Fiyatına göre idare eder, beklentimi karşıladı',
    'Mükemmel bir ürün, çocuğum çok sevdi',
    'Beklediğim gibi çıkmadı, kalitesi düşük',
    'Bu üründen kesinlikle uzak durun, çok kötü',
]

for yorum in demo_yorumlar:
    result = predict_sentiment(yorum, svm_trained)
    print(f"  {result['emoji']} '{yorum[:50]}...' → {result['sentiment']}")

## ✅ Sonuç

Baseline modeller eğitildi ve `models/` klasörüne kaydedildi:
- `models/svm_(linearsvc).joblib`
- `models/naive_bayes.joblib`

**Rapor için kaydedilmesi gereken sayılar:**
- Tablo 4.4.1 (Baseline Sonuçları) — Yukarıdaki `comparison` DataFrame'i
- Şekil 4.4 (Confusion Matrix) — `rapor/sekil_4_4_*.png`

**Sıradaki adım:** `03_transformer_model.ipynb`